In [1]:
import os
import shutil
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torchvision import transforms
from PIL import Image

# Veri yolları
data_dir = Path("../data")
covid_dir = data_dir / "COVID-19_Radiography_Dataset"
xray_dir = data_dir / "chest_xray" / "chest_xray"

print("PyTorch Sürümü:", torch.__version__)
print("CUDA Kullanılabilir mi (GPU):", torch.cuda.is_available())

PyTorch Sürümü: 2.11.0+cu128
CUDA Kullanılabilir mi (GPU): True


In [4]:
covid_verileri = []
siniflar = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]

for sinif in siniflar:
    resim_klasoru = covid_dir / sinif / "images"
    for resim in resim_klasoru.glob("*.png"):
        covid_verileri.append({
            "dosya_yolu": str(resim),
            "sinif": sinif
        })

df_covid = pd.DataFrame(covid_verileri)

# %70 Train, %30 Geçici (Val + Test) - Sınıf oranlarını koruyarak (stratified)
train_df, gecici_df = train_test_split(
    df_covid, test_size=0.30, random_state=42, stratify=df_covid["sinif"]
)

# Kalan %30'u eşit ikiye bölüyoruz -> %15 Val, %15 Test
val_df, test_df = train_test_split(
    gecici_df, test_size=0.50, random_state=42, stratify=gecici_df["sinif"]
)

# Split etiketlerini ekleyelim
train_df = train_df.assign(split="train")
val_df = val_df.assign(split="val")
test_df = test_df.assign(split="test")

df_covid_final = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("--- COVID-19 Radiography Split Dağılımı ---")
print(df_covid_final.groupby(["split", "sinif"]).size().unstack(fill_value=0))
print("\nToplam split sayıları:")
print(df_covid_final["split"].value_counts())

--- COVID-19 Radiography Split Dağılımı ---
sinif  COVID  Lung_Opacity  Normal  Viral Pneumonia
split                                              
test     542           902    1529              202
train   2531          4208    7134              942
val      543           902    1529              201

Toplam split sayıları:
split
train    14815
val       3175
test      3175
Name: count, dtype: int64


In [5]:
xray_verileri = []
bolumler = ["train", "test", "val"]
siniflar2 = ["NORMAL", "PNEUMONIA"]

for bolum in bolumler:
    for sinif in siniflar2:
        klasor = xray_dir / bolum / sinif
        for resim in klasor.glob("*.jpeg"):
            xray_verileri.append({
                "dosya_yolu": str(resim),
                "sinif": sinif,
                "orijinal_split": bolum
            })

df_xray = pd.DataFrame(xray_verileri)

# 1. Test setini olduğu gibi koruyoruz (624 adet)
test_xray = df_xray[df_xray["orijinal_split"] == "test"].copy()
test_xray["split"] = "test"

# 2. train + val birleştirip gerçekçi bir val seti ayırıyoruz (%15 val, %85 train)
train_val_xray = df_xray[df_xray["orijinal_split"].isin(["train", "val"])].copy()

train_xray, val_xray = train_test_split(
    train_val_xray, test_size=0.15, random_state=42, stratify=train_val_xray["sinif"]
)

train_xray = train_xray.assign(split="train")
val_xray = val_xray.assign(split="val")

df_xray_final = pd.concat([train_xray, val_xray, test_xray], ignore_index=True)

print("--- Chest X-Ray Yeni Split Dağılımı ---")
print(df_xray_final.groupby(["split", "sinif"]).size().unstack(fill_value=0))
print("\nToplam split sayıları:")
print(df_xray_final["split"].value_counts())

--- Chest X-Ray Yeni Split Dağılımı ---
sinif  NORMAL  PNEUMONIA
split                   
test      234        390
train    1147       3300
val       202        583

Toplam split sayıları:
split
train    4447
val       785
test      624
Name: count, dtype: int64
